# WorkCover Classification Pipeline

This notebook classifies synthetic employee records to the correct WorkCover code based on Australian state rules.

**ANZSCO** (occupation) codes are predicted via embedding similarity. **ANZSIC** (industry) codes come from the source data (will be replaced by ASIC ABN lookup later).

**State routing logic:**

| Scenario | Primary Driver | Taxonomy |
|---|---|---|
| Standard employer (all states) | Employer's industry | ANZSIC |
| Labour hire — NSW | Worker's **occupation** | ANZSCO |
| Labour hire — VIC, QLD, WA, SA, TAS, ACT | Host employer's **industry** | ANZSIC |

## Setup

In [11]:
# import importlib, subprocess, sys

# def _install(*pkgs):
#     subprocess.check_call([
#         sys.executable, '-m', 'pip', 'install', '-q',
#         '--break-system-packages', *pkgs
#     ])

# _missing = [p for p in ['torch', 'transformers', 'openpyxl'] if importlib.util.find_spec(p) is None]
# if _missing:
#     print(f"Installing: {_missing} ...")
#     _install(*_missing)
#     print("Install complete.")

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

print(f"torch {torch.__version__} | transformers OK")

torch 2.10.0+cu128 | transformers OK


## Ingest Data

In [12]:
# Read ANZSIC codes as strings to preserve any leading zeros (e.g. '0800')
employees = pd.read_csv(
    'Test Data/synthetic_employees.csv',
    dtype={
        'employer_industry_anzsic': str,
        'host_employer_industry_anzsic': str,
        'expected_anzsco_code': str,
    }
)

# Zero-pad ANZSIC codes to 4 digits (class level uses 4-digit codes, some start with 0)
employees['employer_industry_anzsic'] = (
    employees['employer_industry_anzsic']
    .apply(lambda x: str(int(float(x))).zfill(4) if pd.notna(x) and x != 'nan' else None)
)
employees['host_employer_industry_anzsic'] = (
    employees['host_employer_industry_anzsic']
    .apply(lambda x: str(int(float(x))).zfill(4) if pd.notna(x) and x != 'nan' else None)
)

anzsco = pd.read_parquet('ANZSCO/ANZSCO.parquet')
anzsco['Code'] = anzsco['Code'].astype(str).str.strip()

print(f'Employees:  {len(employees):,} rows')
print(f'ANZSCO:     {len(anzsco):,} entries')

Employees:  500 rows
ANZSCO:     1,434 entries


In [13]:
employees.head(10)

,employee_id,raw_job_title,department,state,employer_name,employer_industry_anzsic,employer_industry_name,is_labour_hire,host_employer_name,host_employer_industry_anzsic,host_employer_industry_name,expected_anzsco_code,expected_anzsco_title
0,EMP0001,Enrolled Nurse,Nursing,SA,Adecco,7212,Labour Supply Services,True,Bupa Aged Care,8601,Aged Care Residential,411411,Enrolled Nurse
1,EMP0002,Scrum Master,Agile,NSW,Deloitte Digital,6921,Computer System Design,False,NaN,NaN,NaN,511112,Program or Project Administrator
2,EMP0003,Accounts Payable Clerk,Finance,NSW,IGA Marketplace,1111,Supermarkets,False,NaN,NaN,NaN,551111,Accounts Clerk
3,EMP0004,Data Entry,Operations,QLD,Randstad,7212,Labour Supply Services,True,NAB,6200,Finance,532111,Data Entry Operator
4,EMP0005,Boilermaker/Welder,Manufacturing,VIC,Rio Tinto,0800,Mining Support,False,NaN,NaN,NaN,322311,Metal Fabricator
5,EMP0006,Painter,Construction,VIC,John Holland,3011,Building Construction,False,NaN,NaN,NaN,332211,Painting Trades Worker
6,EMP0007,Wait Staff,Restaurant,NSW,Hays Specialist Recruitment,7212,Labour Supply Services,True,Merivale,9111,Cafes and Restaurants,431511,Waiter
7,EMP0008,Nurse (Registered),Ward 3,VIC,Austin Health,8401,Hospitals,False,NaN,NaN,NaN,254411,Nurse Practitioner
8,EMP0009,Commercial Cleaner,Cleaning,QLD,Randstad,7212,Labour Supply Services,True,Victorian Government,7000,Public Administration,811211,Commercial Cleaner
9,EMP0010,Training Manager,L&D,QLD,University of Queensland,8010,Education,False,NaN,NaN,NaN,223311,Training and Development Professional


In [14]:
anzsco.head()

,Code,Title
0,111111,Chief Executive or Managing Director
1,111211,Corporate General Manager
2,111212,Defence Force Senior Officer
3,111311,Local Government Legislator
4,111312,Member of Parliament


## Step 1 — ANZSCO Classification

Embed each employee's `raw_job_title` using `sentence-transformers/all-MiniLM-L6-v2` and find the closest ANZSCO unit group by cosine similarity.

In [15]:
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
DEVICE = 'cpu'
BATCH_SIZE = 256
MAX_LENGTH = 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

def _mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return (last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

@torch.no_grad()
def embed_texts(texts, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    texts = [str(t) if pd.notna(t) else '' for t in texts]
    all_emb = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        out = model(**enc)
        emb = _mean_pool(out.last_hidden_state, enc['attention_mask'])
        emb = F.normalize(emb, p=2, dim=1)
        all_emb.append(emb.cpu().numpy().astype(np.float32))
    return np.vstack(all_emb)

print('Model loaded.')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 211.42it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.


In [16]:
print('Embedding ANZSCO taxonomy...')
anzsco_embeddings = embed_texts(anzsco['Title'].fillna('').tolist())
print(f'ANZSCO embeddings shape: {anzsco_embeddings.shape}')

Embedding ANZSCO taxonomy...


ANZSCO embeddings shape: (1434, 384)


In [17]:
print('Embedding employee job titles...')
job_embeddings = embed_texts(employees['raw_job_title'].fillna('').tolist())

# Cosine similarity (both sets are already L2-normalised)
similarities = job_embeddings @ anzsco_embeddings.T
best_idx   = similarities.argmax(axis=1)
best_score = similarities[np.arange(len(employees)), best_idx]

employees['predicted_anzsco_code']  = anzsco.iloc[best_idx]['Code'].values
employees['predicted_anzsco_title'] = anzsco.iloc[best_idx]['Title'].values
employees['anzsco_similarity']      = best_score.round(4)

employees[['employee_id', 'raw_job_title', 'predicted_anzsco_code', 'predicted_anzsco_title', 'anzsco_similarity']].head(10)

Embedding employee job titles...


,employee_id,raw_job_title,predicted_anzsco_code,predicted_anzsco_title,anzsco_similarity
0,EMP0001,Enrolled Nurse,411411,Enrolled Nurse,1.0000
1,EMP0002,Scrum Master,231211,Master Fisher,0.4731
2,EMP0003,Accounts Payable Clerk,551111,Accounts Clerk,0.8544
3,EMP0004,Data Entry,532111,Data Entry Operator,0.8085
4,EMP0005,Boilermaker/Welder,322312,Pressure Welder,0.6746
5,EMP0006,Painter,332211,Painter,1.0000
6,EMP0007,Wait Staff,542100,Receptionists nfd,0.4420
7,EMP0008,Nurse (Registered),254418,Registered Nurse (Medical),0.9532
8,EMP0009,Commercial Cleaner,811211,Commercial Cleaner,1.0000
9,EMP0010,Training Manager,149112,Fitness Centre Manager,0.6838


## Step 2 — State-Based WorkCover Routing

Determine which taxonomy drives the WorkCover classification for each employee, and resolve the effective ANZSIC code from the source data:

- **NSW + labour hire** → `occupation` (ANZSCO code drives the WIC)
- **Labour hire, non-NSW** → `industry` (host employer's ANZSIC code from source data)
- **Standard employer** → `industry` (employer's ANZSIC code from source data)

In [18]:
def get_workcover_driver(row):
    """Returns 'occupation' for NSW labour hire, 'industry' for everything else."""
    if bool(row['is_labour_hire']) and str(row['state']).upper() == 'NSW':
        return 'occupation'
    return 'industry'

def get_effective_anzsic_code(row):
    """Resolve the ANZSIC code from source data based on routing rules."""
    if bool(row['is_labour_hire']) and str(row['state']).upper() != 'NSW':
        return row['host_employer_industry_anzsic']
    return row['employer_industry_anzsic']

def get_effective_anzsic_name(row):
    """Resolve the ANZSIC industry name from source data based on routing rules."""
    if bool(row['is_labour_hire']) and str(row['state']).upper() != 'NSW':
        return row['host_employer_industry_name']
    return row['employer_industry_name']

employees['workcover_driver'] = employees.apply(get_workcover_driver, axis=1)
employees['effective_anzsic_code'] = employees.apply(get_effective_anzsic_code, axis=1)
employees['effective_anzsic_name'] = employees.apply(get_effective_anzsic_name, axis=1)

print('Driver breakdown:')
print(employees['workcover_driver'].value_counts())

Driver breakdown:
workcover_driver
industry      434
occupation     66
Name: count, dtype: int64


## Step 3 — Final WorkCover Classification

Combine the routing decision into a single `workcover_code` + `workcover_label` per employee.

In [19]:
def get_workcover_code(row):
    if row['workcover_driver'] == 'occupation':
        return row['predicted_anzsco_code']
    return row['effective_anzsic_code']

def get_workcover_label(row):
    if row['workcover_driver'] == 'occupation':
        return row['predicted_anzsco_title']
    return row['effective_anzsic_name']

employees['workcover_code']  = employees.apply(get_workcover_code, axis=1)
employees['workcover_label'] = employees.apply(get_workcover_label, axis=1)

display_cols = [
    'employee_id', 'raw_job_title', 'state', 'is_labour_hire',
    'workcover_driver', 'workcover_code', 'workcover_label',
]
employees[display_cols]

,employee_id,raw_job_title,state,is_labour_hire,workcover_driver,workcover_code,workcover_label
0,EMP0001,Enrolled Nurse,SA,True,industry,8601,Aged Care Residential
1,EMP0002,Scrum Master,NSW,False,industry,6921,Computer System Design
2,EMP0003,Accounts Payable Clerk,NSW,False,industry,1111,Supermarkets
3,EMP0004,Data Entry,QLD,True,industry,6200,Finance
4,EMP0005,Boilermaker/Welder,VIC,False,industry,0800,Mining Support
...,...,...,...,...,...,...,...
495,EMP0496,Executive Assistant,VIC,False,industry,6200,Finance
496,EMP0497,Forky,NSW,True,occupation,721311,Forklift Driver
497,EMP0498,Digital Marketing Specialist,VIC,False,industry,6921,Computer System Design
498,EMP0499,Senior Accountant,VIC,False,industry,6200,Finance


## Evaluation — ANZSCO Prediction Accuracy

Compare predicted ANZSCO codes against ground-truth labels from the synthetic dataset.

In [20]:
# Only evaluate rows that have an expected ANZSCO code
eval_df = employees[employees['expected_anzsco_code'].notna()].copy()

# Normalise both sides to plain strings for comparison
eval_df['expected_anzsco_code'] = eval_df['expected_anzsco_code'].apply(
    lambda x: str(int(float(x))) if pd.notna(x) else None
)

eval_df['anzsco_correct'] = eval_df['predicted_anzsco_code'] == eval_df['expected_anzsco_code']

accuracy = eval_df['anzsco_correct'].mean()
print(f'ANZSCO Precision@1: {accuracy:.1%}  ({eval_df["anzsco_correct"].sum()} / {len(eval_df)} correct)')
print(f'Mean similarity score: {eval_df["anzsco_similarity"].mean():.4f}')

eval_df[[
    'employee_id', 'raw_job_title', 'state',
    'predicted_anzsco_code', 'predicted_anzsco_title',
    'expected_anzsco_code', 'expected_anzsco_title',
    'anzsco_similarity', 'anzsco_correct',
]]

ANZSCO Precision@1: 33.4%  (167 / 500 correct)
Mean similarity score: 0.7500


,employee_id,raw_job_title,state,predicted_anzsco_code,predicted_anzsco_title,expected_anzsco_code,expected_anzsco_title,anzsco_similarity,anzsco_correct
0,EMP0001,Enrolled Nurse,SA,411411,Enrolled Nurse,411411,Enrolled Nurse,1.0000,True
1,EMP0002,Scrum Master,NSW,231211,Master Fisher,511112,Program or Project Administrator,0.4731,False
2,EMP0003,Accounts Payable Clerk,NSW,551111,Accounts Clerk,551111,Accounts Clerk,0.8544,True
3,EMP0004,Data Entry,QLD,532111,Data Entry Operator,532111,Data Entry Operator,0.8085,True
4,EMP0005,Boilermaker/Welder,VIC,322312,Pressure Welder,322311,Metal Fabricator,0.6746,False
...,...,...,...,...,...,...,...,...,...
495,EMP0496,Executive Assistant,VIC,521111,Personal Assistant,521111,Personal Assistant,0.6771,True
496,EMP0497,Forky,NSW,721311,Forklift Driver,721999,Mobile Plant Operators nec,0.4461,False
497,EMP0498,Digital Marketing Specialist,VIC,225115,Digital Marketing Analyst,225113,Marketing Specialist,0.8585,False
498,EMP0499,Senior Accountant,VIC,221111,Accountant (General),221111,Accountant (General),0.8243,True


In [21]:
# Inspect incorrect predictions
misses = eval_df[~eval_df['anzsco_correct']][
    ['employee_id', 'raw_job_title', 'predicted_anzsco_title', 'expected_anzsco_title', 'anzsco_similarity']
]
print(f'Incorrect predictions: {len(misses)}')
misses

Incorrect predictions: 333


,employee_id,raw_job_title,predicted_anzsco_title,expected_anzsco_title,anzsco_similarity
1,EMP0002,Scrum Master,Master Fisher,Program or Project Administrator,0.4731
4,EMP0005,Boilermaker/Welder,Pressure Welder,Metal Fabricator,0.6746
6,EMP0007,Wait Staff,Receptionists nfd,Waiter,0.4420
7,EMP0008,Nurse (Registered),Registered Nurse (Medical),Nurse Practitioner,0.9532
9,EMP0010,Training Manager,Fitness Centre Manager,Training and Development Professional,0.6838
...,...,...,...,...,...
493,EMP0494,BA,Barista,ICT Business Analyst,0.4705
494,EMP0495,IT Support,ICT Support Engineer,Web Administrator,0.4328
496,EMP0497,Forky,Forklift Driver,Mobile Plant Operators nec,0.4461
497,EMP0498,Digital Marketing Specialist,Digital Marketing Analyst,Marketing Specialist,0.8585


## Export

In [23]:
out_cols = [
    'employee_id', 'raw_job_title', 'department', 'state', 'is_labour_hire',
    'employer_name', 'employer_industry_anzsic', 'employer_industry_name',
    'host_employer_name', 'host_employer_industry_anzsic', 'host_employer_industry_name',
    'predicted_anzsco_code', 'predicted_anzsco_title', 'anzsco_similarity',
    'effective_anzsic_code', 'effective_anzsic_name',
    'workcover_driver', 'workcover_code', 'workcover_label',
]

output = employees[out_cols]
output.to_parquet('Output Data/workcover_classified.parquet', index=False)
output.to_excel('Output Data/workcover_classified.xlsx', index=False)
print(f'Exported {len(output):,} rows.')
output.head()

Exported 500 rows.


,employee_id,raw_job_title,department,state,is_labour_hire,employer_name,employer_industry_anzsic,employer_industry_name,host_employer_name,host_employer_industry_anzsic,host_employer_industry_name,predicted_anzsco_code,predicted_anzsco_title,anzsco_similarity,effective_anzsic_code,effective_anzsic_name,workcover_driver,workcover_code,workcover_label
0,EMP0001,Enrolled Nurse,Nursing,SA,True,Adecco,7212,Labour Supply Services,Bupa Aged Care,8601,Aged Care Residential,411411,Enrolled Nurse,1.0000,8601,Aged Care Residential,industry,8601,Aged Care Residential
1,EMP0002,Scrum Master,Agile,NSW,False,Deloitte Digital,6921,Computer System Design,NaN,NaN,NaN,231211,Master Fisher,0.4731,6921,Computer System Design,industry,6921,Computer System Design
2,EMP0003,Accounts Payable Clerk,Finance,NSW,False,IGA Marketplace,1111,Supermarkets,NaN,NaN,NaN,551111,Accounts Clerk,0.8544,1111,Supermarkets,industry,1111,Supermarkets
3,EMP0004,Data Entry,Operations,QLD,True,Randstad,7212,Labour Supply Services,NAB,6200,Finance,532111,Data Entry Operator,0.8085,6200,Finance,industry,6200,Finance
4,EMP0005,Boilermaker/Welder,Manufacturing,VIC,False,Rio Tinto,0800,Mining Support,NaN,NaN,NaN,322312,Pressure Welder,0.6746,0800,Mining Support,industry,0800,Mining Support
